# WC 2026 Predictions — Hybrid ELO + Pi-Ratings GLM

Generates score predictions for all unplayed WC 2026 group-stage matches using the best validated model (Exp D, 4.387 pts/match LOTO-CV).

**Model:** Poisson GLM with hybrid ELO + pi-ratings features, retrained on the full training set (all matches up to 2026-06-01).  
**Score strategy:** predict (h, a) that maximises expected Sporza points under the Poisson joint distribution.  
**Output:** `data/predictions/wc2026_predictions.csv`

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
from functools import lru_cache
from pathlib import Path
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from penaltyblog.ratings import PiRatingSystem

DATA    = Path('../../data/processed')
INTERIM = Path('../../data/interim')
OUT_DIR = Path('../../data/predictions')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Cutoff: all matches up to (but not including) WC 2026 start
WC2026_CUTOFF = '2026-06-11'
MAX_GOALS = 8

# Exact feature order used in Exp D (notebook 19)
HOME_FEATURES = [
    'home_elo', 'away_elo', 'elo_diff',
    'pi_home', 'pi_away', 'pi_diff',
    'is_neutral',
    'home_form_wr', 'away_form_wr',
    'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga',
    'tournament_weight',
]
AWAY_FEATURES = [
    'away_elo', 'home_elo', 'elo_diff',
    'pi_away', 'pi_home', 'pi_diff',
    'is_neutral',
    'away_form_wr', 'home_form_wr',
    'away_form_gf', 'home_form_gf',
    'away_form_ga', 'home_form_ga',
    'tournament_weight',
]

print('Paths OK.')
print(f'Output directory: {OUT_DIR.resolve()}')

Paths OK.
Output directory: /Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/data/predictions


## 1. Load data

In [2]:
# Full training set (all matches before WC 2026)
train = pd.read_parquet(DATA / 'train.parquet')
print(f'Training set: {len(train)} matches  ({train.date.min().date()} → {train.date.max().date()})')

# WC 2026 feature table (has ELO + form, no pi)
wc_feat = pd.read_parquet(INTERIM / 'wc2026_features.parquet')
print(f'WC 2026 features: {len(wc_feat)} matches  ({wc_feat.date.min().date()} → {wc_feat.date.max().date()})')

# Full match history for pi-rating computation
matches_full = pd.read_parquet(INTERIM / 'matches.parquet')
print(f'Full match history: {len(matches_full)} matches  ({matches_full.date.min().date()} → {matches_full.date.max().date()})')

# Separate played vs unplayed
unplayed = wc_feat[wc_feat['played'] == False].copy()
played   = wc_feat[wc_feat['played'] == True].copy()
print(f'\nUnplayed matches to predict: {len(unplayed)}')
print(f'Already played (skipping):   {len(played)}')

Training set: 11106 matches  (2010-01-02 → 2022-10-30)
WC 2026 features: 72 matches  (2026-06-11 → 2026-06-27)
Full match history: 32288 matches  (1990-01-12 → 2026-06-11)

Unplayed matches to predict: 70
Already played (skipping):   2


## 2. Compute pi-ratings up to WC 2026 cutoff

In [3]:
# Same hyperparameters as in Exp C/D notebooks
pi = PiRatingSystem(alpha=0.15, beta=0.10, k=0.75)

training_matches = matches_full[matches_full['date'] < WC2026_CUTOFF].sort_values('date')
print(f'Fitting pi-ratings on {len(training_matches)} matches up to {WC2026_CUTOFF}...')

for _, row in training_matches.iterrows():
    goal_diff = int(row['home_score'] - row['away_score'])
    pi.update_ratings(row['home_team'], row['away_team'], goal_diff)

team_ratings = pi.team_ratings
print(f'Pi-ratings computed for {len(team_ratings)} teams.')

# Sanity check on top nations
top = sorted(team_ratings.items(), key=lambda x: x[1]['home'], reverse=True)[:10]
print('\nTop 10 teams by pi_home:')
for team, r in top:
    print(f'  {team:<25}  home={r["home"]:+.3f}  away={r["away"]:+.3f}')

Fitting pi-ratings on 32286 matches up to 2026-06-11...


Pi-ratings computed for 326 teams.

Top 10 teams by pi_home:
  Spain                      home=+3.333  away=+2.844
  Argentina                  home=+3.248  away=+2.998
  Brazil                     home=+2.860  away=+2.691
  France                     home=+2.819  away=+2.631
  Belgium                    home=+2.815  away=+2.235
  Portugal                   home=+2.784  away=+2.290
  Japan                      home=+2.754  away=+2.531
  Germany                    home=+2.737  away=+2.472
  Ecuador                    home=+2.691  away=+2.205
  Colombia                   home=+2.606  away=+2.870


In [4]:
def add_pi_features(df, ratings):
    df = df.copy()
    df['pi_home'] = df['home_team'].map(lambda t: ratings[t]['home'] if t in ratings else 0.0)
    df['pi_away'] = df['away_team'].map(lambda t: ratings[t]['away'] if t in ratings else 0.0)
    df['pi_diff'] = df['pi_home'] - df['pi_away']
    return df

train    = add_pi_features(train, team_ratings)
unplayed = add_pi_features(unplayed, team_ratings)

# Report any teams with no pi-rating
missing_home = unplayed[unplayed['pi_home'] == 0.0]['home_team'].unique()
missing_away = unplayed[unplayed['pi_away'] == 0.0]['away_team'].unique()
if len(missing_home) or len(missing_away):
    print('Teams defaulting to pi=0.0 (not in history):', set(missing_home) | set(missing_away))
else:
    print('All WC 2026 teams have pi-ratings. No fallback needed.')

All WC 2026 teams have pi-ratings. No fallback needed.


## 3. Train hybrid GLM on full training set

In [5]:
def build_X(df, features):
    X = df[features].copy().fillna(df[features].median())
    return X.values

X_home = build_X(train, HOME_FEATURES)
X_away = build_X(train, AWAY_FEATURES)
X_tr   = np.vstack([X_home, X_away])
y_tr   = np.concatenate([train['home_score'].values, train['away_score'].values])

pipe = Pipeline([
    ('scaler',  StandardScaler()),
    ('poisson', PoissonRegressor(alpha=0.1, max_iter=300)),
])
pipe.fit(X_tr, y_tr)
print(f'Model trained on {len(X_tr)} rows ({len(train)} matches × 2 sides).')
print(f'Converged: {pipe.named_steps["poisson"].n_iter_} iterations')

Model trained on 22212 rows (11106 matches × 2 sides).
Converged: 15 iterations


## 4. Generate predictions

In [6]:
@lru_cache(maxsize=50000)
def best_score_cached(lh_r, la_r):
    """(h, a) maximising expected Sporza pts given Poisson(lh_r) × Poisson(la_r)."""
    goals   = np.arange(MAX_GOALS + 1)
    ph_pmf  = poisson.pmf(goals, lh_r)
    pa_pmf  = poisson.pmf(goals, la_r)
    joint   = np.outer(ph_pmf, pa_pmf)

    def expected_pts(pred_h, pred_a):
        pts = 0.0
        for ah in range(MAX_GOALS + 1):
            for aa in range(MAX_GOALS + 1):
                p = joint[ah, aa]
                if p < 1e-9:
                    continue
                if pred_h == ah and pred_a == aa:
                    pts += p * 10
                elif (pred_h - pred_a) == (ah - aa):
                    pts += p * 7
                elif np.sign(pred_h - pred_a) == np.sign(ah - aa):
                    pts += p * 5
                else:
                    pts += p * 1
        return pts

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph in range(MAX_GOALS + 1):
        for pa in range(MAX_GOALS + 1):
            ep = expected_pts(ph, pa)
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph, pa
    return best_h, best_a, round(best_pts, 4)


lh = pipe.predict(build_X(unplayed, HOME_FEATURES))
la = pipe.predict(build_X(unplayed, AWAY_FEATURES))

preds = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]

predictions = unplayed[['date', 'home_team', 'away_team', 'group', 'tournament']].copy()
predictions['lambda_home'] = lh.round(3)
predictions['lambda_away'] = la.round(3)
predictions['pred_home']   = [p[0] for p in preds]
predictions['pred_away']   = [p[1] for p in preds]
predictions['exp_pts']     = [p[2] for p in preds]

print(f'Generated predictions for {len(predictions)} matches.')
print(f'Mean expected Sporza pts: {predictions["exp_pts"].mean():.3f}')

Generated predictions for 70 matches.
Mean expected Sporza pts: 4.181


In [7]:
print(predictions[['date', 'home_team', 'away_team', 'pred_home', 'pred_away',
                    'lambda_home', 'lambda_away', 'exp_pts']].to_string(index=False))

      date            home_team            away_team  pred_home  pred_away  lambda_home  lambda_away  exp_pts
2026-06-18       Czech Republic         South Africa          1          0        1.692        0.840   4.2166
2026-06-18               Mexico          South Korea          1          0        1.487        0.872   3.9998
2026-06-24       Czech Republic               Mexico          0          1        0.929        1.438   3.8552
2026-06-24         South Africa          South Korea          0          1        0.839        1.435   3.9922
2026-06-12               Canada Bosnia & Herzegovina          1          0        1.604        0.733   4.3277
2026-06-13                Qatar          Switzerland          0          2        0.603        2.274   4.8983
2026-06-18          Switzerland Bosnia & Herzegovina          1          0        1.927        0.778   4.4691
2026-06-18               Canada                Qatar          2          0        2.427        0.540   5.0770
2026-06-24

## 5. Prediction distribution analysis

In [8]:
score_counts = predictions.groupby(['pred_home', 'pred_away']).size().reset_index(name='count').sort_values('count', ascending=False)
print('Most common predicted scorelines:')
print(score_counts.head(10).to_string(index=False))

print(f'\nLambda stats:')
print(f'  lambda_home: mean={lh.mean():.3f}  min={lh.min():.3f}  max={lh.max():.3f}')
print(f'  lambda_away: mean={la.mean():.3f}  min={la.min():.3f}  max={la.max():.3f}')

win  = (predictions['pred_home'] > predictions['pred_away']).sum()
draw = (predictions['pred_home'] == predictions['pred_away']).sum()
loss = (predictions['pred_home'] < predictions['pred_away']).sum()
print(f'\nOutcome distribution: home win {win}  draw {draw}  away win {loss}  (total {len(predictions)})')

Most common predicted scorelines:
 pred_home  pred_away  count
         1          0     33
         0          1     26
         2          0      7
         0          2      2
         3          0      2

Lambda stats:
  lambda_home: mean=1.524  min=0.569  max=3.559
  lambda_away: mean=1.111  min=0.465  max=2.612

Outcome distribution: home win 42  draw 0  away win 28  (total 70)


## 6. Save predictions

In [9]:
out_path = OUT_DIR / 'wc2026_predictions.csv'
predictions.to_csv(out_path, index=False)
print(f'Saved {len(predictions)} predictions → {out_path.resolve()}')

Saved 70 predictions → /Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/data/predictions/wc2026_predictions.csv
